## ANA - <font color=purple>*in vivo*</font> - Analysis of Ectoderm/Myeloid Cell Movements

### Notes

#### Information on input

- _Per-track main data_ (`Ectoderm-data_clean_vec.pkl`, `Myeloid-data_clean_vec.pkl`)
    - `ecto_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `myel_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `*` Details on df columns:
        - `f, t, y, x`: frames `[1]`, times `[min]`, positions `[microns]`
        - ~`cen_y, cen_x, y_rel, x_rel, r, clust_r`~: cluster center, relative positions, radial distances, cluster size `[microns]`
        - `vy, vx, vy_itp, vx_itp`: velocities, locally interpolated ectoderm velocities `[microns/min]`
        - `v_mag, v_itp_mag`: velocity vector magnitudes
        - `vy_n, vx_n, vy_itp_n, vx_itp_n`: components of normalized (magnitude 1.0) vectors
        - `vy_traj, vx_traj, vy_traj_n, vx_traj_n`: cell track trajectory-aligned (optionally normalized) vectors
        - ~`vy_ray, vx_ray, vy_ray_n, vx_ray_n`~: cluster radial axis-aligned (optionally normalized) vectors

* _Per-track similarity metrics_ (`Ectoderm-similarities.pkl`, `Myeloid-similarities.pkl`)
    - `ecto_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
    - `myel_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
    - `*` Details on df columns:
        - `f, t, y, x,` ~`cen_y, cen_x, y_rel, x_rel, r, clust_r`~: as in `ecto_data` and `myel_data`
        - `metric + "-shift="+str(s) for s in profile_time_shifts`: Per-cell local similarity for each similarity metric

- _Time-shift profile data per frame_ (`Ectoderm-profiles.pkl`, `Myeloid-profiles.pkl`)
    - `ecto_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`
    - `myel_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`

* _Time-shift profile data averaged over frames_ (`Ectoderm-profiles_mean.pkl`, `Myeloid-profiles_mean.pkl`)
    - `ecto_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`
    - `myel_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`

- Conditions: `CT` means control (wild type), `DD1` is a Wnt dominant negative where the embryos do not grow
    - _Note that there is not enough data to conclude anything for DD1, and even conceptually this analysis may not apply there!_
    
* Tracking was done with StarDist for the ectoderm and manually in ImageJ for myeloid cells

- <font color=purple>_In vivo_ note:</font> Samples are collected from two experiments:
    - Samples with suffix `A`, as in `Pos(ddd)A` are a subset from experiment `20260201`
    - Samples without a suffix, as in `Pos(ddd)` are a subset from experiment `20260222`
    - Included samples were selected based on whether they have a sufficient number of sufficiently long myeloid tracks


#### Pre-requisites

- The data must have been preprocessed with `RUN - in vivo - 1 - Preprocessing.ipynb`
- Movement vectors must have been extracted and interpolated with `RUN - in vivo - 2 - Movement vectors.ipynb`
- Shifted time profiles must have been computed with `RUN - in vivo - 3 - Correlations and similarities.ipynb`


#### Content of this notebook

1. Set parameters and load the data
2. Interactive data visualizations from different pipeline steps
3. Remove problematic outliers identified as unhealthy explants (not applicable in vivo)
4. Analysis of correlation/similarity profiles over time shifts
5. Ectoderm-myeloid transfer functions for cue saturation model

### Prep

In [ ]:
### Imports

%load_ext autoreload
%autoreload 2

import os, warnings, pickle
from IPython.display import display

import itertools
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors as mplcolors
import ipywidgets
from ipywidgets import interact

from scipy.spatial import distance as spdist
from scipy import stats
from scipy.optimize import curve_fit

import sys; sys.path.insert(0, '..')
from tracking_analysis.utilities import savebutton, get_shifted_data
import tracking_analysis.pcl_tools as pcl_tools

In [ ]:
### Seeding

np.random.seed(42)

In [ ]:
### Parameters

# Units
pxl_res  = 1.1375  # [microns]
time_res = 5       # [min]

# Shifted time profiles
prof_min_shift  = -int(30*5/time_res) # Range of profile into past  # Adjusted to approx. same range as ex vivo
prof_max_shift  =  int(30*5/time_res) # Range of profile into future  # Adjusted to approx. same range as ex vivo
prof_mean_range = pmr = (70, 140)  # Time point range for averaged profile  # Adjusted for in vivo

# Whether to flip profile shift sign
# Note: Initially, the corr/sim profiles were thought of as being computed "against the ectoderm as a reference", 
#       so negative shifts meaning comparisons to past ectoderm (and vice versa) made good sense. However, after 
#       the myeloid's negative-shifted peak was found, the point was made that it would make more sense to flip 
#       the shifts, thinking of the myel peak as "occurring after" the ecto peak along a left-to-right axis in 
#       time. This is implemented if below flag is set to true.
flip_shifts = True

# Condition labels dictionary
# Note: There's not enough data to conclude anything for DD1, and even conceptually 
#       this analysis may not apply there, so it is not included in the figures.
cond_dict = {'CT' : 'Control (wild type)', 'DD1' : 'DD1 dominant negative'}

# Optimized y-ranges for different metrics
yranges_tight = {
    "corr"              : ( 0.00, 0.85),
    "corr_n"            : ( 0.00, 0.80),
    #"speed_corr"        : (-0.15, 0.75),
    "speed_corr"        : (-0.20, 0.50),  # Updated for in vivo
    "traj_para_corr"    : (-0.20, 0.85),
    "traj_para_n_corr"  : (-0.20, 0.85),
    "traj_ortho_corr"   : (-0.20, 0.85),
    "traj_ortho_n_corr" : (-0.20, 0.85),
    #"ray_para_corr"     : (-0.20, 0.90),  # N/A in vivo
    #"ray_para_n_corr"   : (-0.20, 0.90),  # N/A in vivo
    #"ray_ortho_corr"    : (-0.10, 0.85),  # N/A in vivo
    #"ray_ortho_n_corr"  : (-0.10, 0.85),  # N/A in vivo
    #"cos_sim"           : ( 0.10, 0.85),
    "cos_sim"           : (-0.20, 1.00),  # Updated for in vivo
    "speed_sim"         : ( 0.60, 0.85),
    "traj_para_sim"     : ( 0.60, 0.87),
    "traj_para_n_sim"   : ( 0.60, 0.87),
    "traj_ortho_sim"    : ( 0.60, 0.87),
    "traj_ortho_n_sim"  : ( 0.60, 0.87),
    #"ray_para_sim"      : ( 0.60, 0.90),  # N/A in vivo
    #"ray_para_n_sim"    : ( 0.60, 0.90),  # N/A in vivo
    #"ray_ortho_sim"     : ( 0.60, 0.87),  # N/A in vivo
    #"ray_ortho_n_sim"   : ( 0.60, 0.87),  # N/A in vivo
}

yranges_limits = {
    "corr"              : (-1.0, 1.0),
    "corr_n"            : (-1.0, 1.0),
    "speed_corr"        : (-1.0, 1.0),
    "traj_para_corr"    : (-1.0, 1.0),
    "traj_para_n_corr"  : (-1.0, 1.0),
    "traj_ortho_corr"   : (-1.0, 1.0),
    "traj_ortho_n_corr" : (-1.0, 1.0),
    #"ray_para_corr"     : (-1.0, 1.0),  # N/A in vivo
    #"ray_para_n_corr"   : (-1.0, 1.0),  # N/A in vivo
    #"ray_ortho_corr"    : (-1.0, 1.0),  # N/A in vivo
    #"ray_ortho_n_corr"  : (-1.0, 1.0),  # N/A in vivo
    "cos_sim"           : (-1.0, 1.0),
    "speed_sim"         : ( 0.0, 1.0),
    "traj_para_sim"     : ( 0.0, 1.0),
    "traj_para_n_sim"   : ( 0.0, 1.0),
    "traj_ortho_sim"    : ( 0.0, 1.0),
    "traj_ortho_n_sim"  : ( 0.0, 1.0),
    #"ray_para_sim"      : ( 0.0, 1.0),  # N/A in vivo
    #"ray_para_n_sim"    : ( 0.0, 1.0),  # N/A in vivo
    #"ray_ortho_sim"     : ( 0.0, 1.0),  # N/A in vivo
    #"ray_ortho_n_sim"   : ( 0.0, 1.0),  # N/A in vivo
}

In [ ]:
### Data locations

top_path = r"..\Data\in_vivo"

ecto_base = r"Ectoderm"
myel_base = r"Myeloid"

In [ ]:
### Load the data

# Ectoderm
with open(os.path.join(top_path, ecto_base+'-data_clean_vec_prof.pkl'), "rb") as infile:
    ecto_data = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+"-similarities.pkl"), "rb") as infile:
    ecto_similarities = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+'-profiles.pkl'), "rb") as infile:
    ecto_profiles = pickle.load(infile)
with open(os.path.join(top_path, ecto_base+'-profiles_mean.pkl'), "rb") as infile:
    ecto_profs_mean = pickle.load(infile)

# Myeloid
with open(os.path.join(top_path, myel_base+'-data_clean_vec_prof.pkl'), "rb") as infile:
    myel_data = pickle.load(infile)
with open(os.path.join(top_path, myel_base+"-similarities.pkl"), "rb") as infile:
    myel_similarities = pickle.load(infile)
with open(os.path.join(top_path, myel_base+'-profiles.pkl'), "rb") as infile:
    myel_profiles = pickle.load(infile)
with open(os.path.join(top_path, myel_base+'-profiles_mean.pkl'), "rb") as infile:
    myel_profs_mean = pickle.load(infile)

# Report
print("\nEctoderm data:\n")
print(' ', ecto_data['CT'].keys())
print(' ', ecto_data['DD1'].keys())
print(' ', ecto_data['CT'][list(ecto_data['CT'].keys())[0]].shape)
print(' ', ecto_profiles['CT'][list(ecto_data['CT'].keys())[0]].keys())
print(' ', ecto_profiles['CT'][list(ecto_data['CT'].keys())[0]]['cos_sim'].shape)
print(' ', ecto_profs_mean['CT'][list(ecto_data['CT'].keys())[0]]['cos_sim'].shape)

print("\nMyeloid data:\n")
print(' ', myel_data['CT'].keys())
print(' ', myel_data['DD1'].keys())
print(' ', myel_data['CT'][list(ecto_data['CT'].keys())[0]].shape)
print(' ', myel_profiles['CT'][list(ecto_data['CT'].keys())[0]].keys())
print(' ', myel_profiles['CT'][list(ecto_data['CT'].keys())[0]]['cos_sim'].shape)
print(' ', myel_profs_mean['CT'][list(ecto_data['CT'].keys())[0]]['cos_sim'].shape)

In [ ]:
### Extract some useful bits and pieces

# Lists of available metrics
metrics = list(ecto_profs_mean['CT'][list(ecto_data['CT'].keys())[0]].keys())
metrics_similarity = [m for m in metrics if m.endswith("_sim")]
metrics_correlation = [m for m in metrics if (m.startswith("corr") or m.endswith("_corr"))]
print("Available metrics:", metrics)
print("\n...of which are (local) similarity metrics:", metrics_similarity)
print("\n...of which are (global) correlation metrics:", metrics_correlation)

# Overall first and last frame in myeloid
minmax_frames_myel = [np.inf, -np.inf]
for c in myel_data.keys():
    for p in myel_data[c].keys():
        if myel_data[c][p]['f'].min() < minmax_frames_myel[0]:
            minmax_frames_myel[0] = myel_data[c][p]['f'].min()
        if myel_data[c][p]['f'].max() > minmax_frames_myel[1]:
            minmax_frames_myel[1] = myel_data[c][p]['f'].max()
print('\nOverall first & last frame in myel data:', minmax_frames_myel)

In [ ]:
### Flip shifts if desired (see note in `parameters` code cell above)

if flip_shifts:   
    
    # Flip parameters
    prof_min_shift  = -prof_min_shift  # Time point range of profile into past
    prof_max_shift  = -prof_max_shift  # Time point ange of profile into future
    
    # Flip data indexers
    for c in ecto_data.keys():
        for p in ecto_data[c].keys():
            
            # Flip similarities df columns
            flipped_cols = [
                c.split("-shift=")[0] + "-shift=" + str(-int(c.split("-shift=")[-1])) 
                if "-shift=" in c else c
                for c in myel_similarities[c][p].columns
            ]
            ecto_similarities[c][p].columns = flipped_cols
            myel_similarities[c][p].columns = flipped_cols
            
            # Flip profiles df columns
            for metric in ecto_profiles[c][p].keys():
                ecto_profiles[c][p][metric].columns = -ecto_profiles[c][p][metric].columns
                myel_profiles[c][p][metric].columns = -myel_profiles[c][p][metric].columns
                
            # Flip mean profiles series indices
            for metric in ecto_profs_mean[c][p].keys():
                ecto_profs_mean[c][p][metric].index = -ecto_profs_mean[c][p][metric].index
                myel_profs_mean[c][p][metric].index = -myel_profs_mean[c][p][metric].index

### Interactive visualizations compiled from processing pipeline

In [ ]:
### Visualize the data

@interact(show=False, condition=ecto_data.keys())
def show_by_condition(show=False, condition='CT'):
    
    if not show:
        return
    
    @interact(position=ecto_data[condition].keys(),
              ecto_fraction=['20%', '0%', '10%', '20%', '40%', '100%'],
              show_overlay=True)
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0],
                         ecto_fraction='20%',
                         show_overlay=True):
        
        # Select data
        ecto_df = ecto_data[condition][position]
        myel_df = myel_data[condition][position]
        
        # Handle overlay vs separate subplots
        if show_overlay:
            fig, ax = plt.subplots(1, figsize=(7, 7))
            ax = [ax, ax]
        else:
            fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
        
        # Plot ectoderm tracks
        if ecto_fraction != '0%':
            for TID in ecto_df.index.unique()[::100//int(ecto_fraction[:-1])]:
                ax[0].scatter(
                    ecto_df.loc[ecto_df.index==TID, 'x'],
                    ecto_df.loc[ecto_df.index==TID, 'y'],
                    c=ecto_df.loc[ecto_df.index==TID, 'f'],
                    cmap='winter_r', s=5, alpha=0.3)
                
        # Plot myeloid tracks
        for TID in myel_df.index.unique():
            ax[1].scatter(
                myel_df.loc[myel_df.index==TID, 'x'],
                myel_df.loc[myel_df.index==TID, 'y'],
                c=myel_df.loc[myel_df.index==TID, 'f'],
                cmap='autumn_r', s=15, alpha=0.7)
        
        # Cosmetics
        for axis in ax:
            axis.axis('equal')
            axis.axis('off')
        
        # Finalize
        plt.gca().invert_yaxis()
        plt.tight_layout()

In [ ]:
### Visualize interpolated vectors

@interact(show=False, condition=ecto_data.keys())
def show_by_condition(show=False, condition='CT'):
    
    if not show:
        return
    
    @interact(position=ecto_data[condition].keys())
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        @interact(frame=(0, ecto_data[condition][position]['f'].max(), 1), 
                  scale=(0.01, 0.1, 0.01), quadrant=False)
        @savebutton
        def show_vectors(frame=ecto_data[condition][position]['f'].max()//2, 
                         scale=0.15, quadrant=False):
            
            # Select relevant data
            ecto_df = ecto_data[condition][position]
            myel_df = myel_data[condition][position]
            
            # Create frame masks
            ecto_fmask = (ecto_df['f'] == frame).values
            myel_fmask = (myel_df['f'] == frame).values
            myel_fmask_p1 = (myel_df['f'] == frame+1).values

            # Prep figure
            fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)

            # Plot observed vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'],  ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx'], ecto_df.loc[ecto_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'],  myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx'], myel_df.loc[myel_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)

            # Plot interpolated vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx_itp'], ecto_df.loc[ecto_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale,
                         color='red', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx_itp'], myel_df.loc[myel_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='red', alpha=0.6, width=0.003)

            # Plot source points
            ax[0].scatter(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)
            ax[1].scatter(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)

            # Axis limits
            if quadrant:
                ax[0].set_xlim([1000, 1600])
                ax[0].set_ylim([ 800, 1400])

            # Labels
            ax[0].set_title(f'Ectoderm (t={frame*time_res}min)')
            ax[1].set_title(f'Myeloid (t={frame*time_res}min)')

            # Finish
            plt.tight_layout()

In [ ]:
### Visualize the resulting correlation/similarity time profiles

@interact(condition=ecto_data.keys())
def show_by_condition(condition='CT'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        # Select relevant data
        ecto_profs = ecto_profiles[condition][position]
        myel_profs = myel_profiles[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            2, len(ecto_profs.keys()), 
            figsize=(1.1*len(ecto_profs.keys()), 6), 
            sharex=True, sharey=True)

        # For each metric...
        for m, metric in enumerate(ecto_profs.keys()):

            # Generate the heatmap
            ax[0,m].imshow(ecto_profs[metric], interpolation='none')
            ax[1,m].imshow(myel_profs[metric], interpolation='none')

            # Set cosmetics
            ax[0,m].set_title(metric, fontsize=8, rotation=80, ha="left")
            ax[1,m].set_xlabel('$shift$')
            if not flip_shifts:
                ax[1,m].set_xticks([0, (prof_max_shift-prof_min_shift)//2, prof_max_shift-prof_min_shift])
                ax[1,m].set_xticklabels([prof_min_shift, prof_min_shift+(prof_max_shift-prof_min_shift)//2, prof_max_shift])
            else:
                ax[1,m].set_xticks([0, (prof_min_shift-prof_max_shift)//2, prof_min_shift-prof_max_shift])
                ax[1,m].set_xticklabels([prof_min_shift, prof_min_shift+(prof_max_shift-prof_min_shift)//2, prof_max_shift])
            
        # Since indexers (not data) are flipped, imshow must be flipped as well!
        ax[0,m].invert_xaxis()

        # More cosmetics
        ax[0,0].set_ylabel('Ectoderm\n\n$frame$')
        ax[1,0].set_ylabel('Myeloid\n\n$frame$')

        # Finalize
        plt.tight_layout()

In [ ]:
### Visualize the resulting *averaged* correlation/similarity time profiles (per sample)

# Weird hack to prevent autoscrolling of widget output
style = """
    <style>
       .jupyter-widgets-output-area .output_scroll {
            height: unset !important;
            border-radius: unset !important;
            -webkit-box-shadow: unset !important;
            box-shadow: unset !important;
        }
        .jupyter-widgets-output-area  {
            height: auto !important;
        }
    </style>
    """
display(ipywidgets.HTML(style))

@interact(condition=ecto_data.keys())
def show_by_condition(condition='CT'):
    
    @interact(position=ecto_data[condition].keys(),
              yrange=["tight", "limits", "indiv"])
    @savebutton
    def show_by_position(
        position=list(ecto_data[condition].keys())[0], 
        yrange="tight"):
        
        # Select relevant data
        ecto_pms = ecto_profs_mean[condition][position]
        myel_pms = myel_profs_mean[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            int(np.ceil(len(ecto_pms.keys())/2)), 2, 
            figsize=(8, 2*int(np.ceil(len(ecto_pms.keys())/2))), 
            sharex=True, sharey=yrange=="common")
        if len(ecto_pms.keys()) % 2 != 0:
            ax[-1, -1].set_visible(False)

        # For each metric...
        for m, metric in enumerate(ecto_pms.keys()):
            
            # Plot profiles
            ax[m//2, m%2].plot(
                ecto_pms[metric].index * time_res, 
                ecto_pms[metric], 
                label='Ectoderm')
            ax[m//2, m%2].plot(
                myel_pms[metric].index * time_res, 
                myel_pms[metric], 
                label='Myeloid')

            # Set cosmetics
            ax[m//2, m%2].set_title(f"{condition} - {position} - {metric}")
            if (m//2) == 2:
                ax[m//2, m%2].set_xlabel('time shift [min]')
            if (m%2) == 0:
                ax[m//2, m%2].set_ylabel('metric')
                
            # Set y ranges
            if yrange == "tight":
                ax[m//2, m%2].set_ylim(yranges_tight[metric])
            if yrange == "limits":
                ax[m//2, m%2].set_ylim(yranges_limits[metric])
            
        # Crop x a bit to visualize peaks better
        plt.xlim(-100, 100)

        # Add midlines...
        for m in range(ax.size):
            ymin, ymax = ax[m//2, m%2].get_ylim()
            ax[m//2, m%2].vlines(0, ymin, ymax, color='k', lw=0.5, alpha=0.3, zorder=-1)
            ax[m//2, m%2].set_ylim(ymin, ymax)

        # Finalize
        plt.tight_layout()

### Remove problematic outliers identified as unhealthy explants

In [ ]:
### Outlier removal

# - There are no samples that have been identified as outliers in the in vivo dataset

excluded_outliers = []
#excluded_outliers = [("norm", "Pos007"), ]  # Example from ex vivo

for outlier in excluded_outliers:
    del ecto_data[outlier[0]][outlier[1]]
    del myel_data[outlier[0]][outlier[1]]
    del ecto_similarities[outlier[0]][outlier[1]]
    del myel_similarities[outlier[0]][outlier[1]]
    del ecto_profiles[outlier[0]][outlier[1]]
    del myel_profiles[outlier[0]][outlier[1]]
    del ecto_profs_mean[outlier[0]][outlier[1]]
    del myel_profs_mean[outlier[0]][outlier[1]]

### Analysis of correlation/similarity profiles over time shifts

In [ ]:
### Function to calculate profile stats (avg & std) across samples

def get_profile_stats(profs_mean, condition, metric):
    
    c = condition
    
    compiled = [profs_mean[c][p][metric] for p in profs_mean[c].keys()]
    compiled = np.array(compiled)
    
    avg = pd.Series(
        np.mean(compiled, axis=0),
        #np.nanmean(compiled, axis=0),  # Would be needed for DD1 due to limited data
        index=profs_mean[c][list(profs_mean[c].keys())[0]][metric].index,
        name='< '+condition+' - '+metric+' - mean >')
    std = pd.Series(
        np.std(compiled, axis=0),
        #np.nanmean(compiled, axis=0),  # Would be needed for DD1 due to limited data
        index=profs_mean[c][list(profs_mean[c].keys())[0]][metric].index,
        name='< '+condition+' - '+metric+' - std >')
    
    return avg, std

In [ ]:
### Function to visualize correlation/similarity profiles across samples

def show_profile_single(
    ecto_profs_mean, myel_profs_mean, metric, 
    condition="CT", yrange="tight",
    show_samples=True, show_avg=True):
    
    # Prep
    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    ccycler = plt.rcParams["axes.prop_cycle"].by_key()['color']
    
    # Adapt for single plot
    ax = [ax]
    c = condition
    axi = 0
    
    # Plot individual profiles
    if show_samples:
        for p in ecto_profs_mean[c].keys():
            ax[axi].plot(
                ecto_profs_mean[c][p][metric].index * time_res, 
                ecto_profs_mean[c][p][metric], 
                color=ccycler[0], lw=1, alpha=0.2)
            ax[axi].plot(
                myel_profs_mean[c][p][metric].index * time_res, 
                myel_profs_mean[c][p][metric], 
                color=ccycler[1], lw=1, alpha=0.2)
        
    # Plot average profiles
    if show_avg:
        ecto_avg, ecto_std = get_profile_stats(ecto_profs_mean, c, metric)
        myel_avg, myel_std = get_profile_stats(myel_profs_mean, c, metric)
        ax[axi].plot(
            ecto_avg.index * time_res, ecto_avg,
            color=ccycler[0], lw=2, alpha=0.8,
            label='Ectoderm')
        ax[axi].plot(
            myel_avg.index * time_res, myel_avg,
            color=ccycler[1], lw=2, alpha=0.8,
            label='Myeloid')
        
    # Set cosmetics
    ax[0].legend(fontsize=8, frameon=False)
    ax[0].set_xlabel('time shift [min]')
    ax[axi].set_title(f'{cond_dict[c]}', fontsize=10.5)
    pub_ylbls = {
        "cos_sim" : "cosine similarity", "speed_corr" : "correlation of speeds",
        "traj_ortho_corr" : "correlation of\ntrajectory deviations"}
    ax[0].set_ylabel(pub_ylbls[metric] if metric in pub_ylbls else metric)
    
    # Axis settings
    plt.xlim(-100, 100)     # Crop x a bit to visualize peaks better
    if yrange == "tight":
        ax[0].set_ylim(yranges_tight[metric])
    if yrange == "limits":
        ax[0].set_ylim(yranges_limits[metric])

    # Add midlines...
    ymin, ymax = ax[axi].get_ylim()
    ax[axi].vlines(0, ymin, ymax, color='k', lw=0.5, alpha=0.3, zorder=-1)
    ax[axi].set_ylim(ymin, ymax)
        
    # IN VIVO: Show tick labels on top plot because bottom plot is not of interest
    ax[0].xaxis.set_tick_params(labelbottom=True)

    # Finalize
    plt.tight_layout()

In [ ]:
### Show the correlation/similarity profiles

@interact(
    metric=metrics[2:], 
    condition=list(ecto_profs_mean.keys()), 
    yrange=["tight", "limits", "indiv"])
@savebutton
def wrapper(metric="traj_ortho_corr", condition="CT", yrange="tight"):
    show_profile_single(
        ecto_profs_mean, myel_profs_mean, metric, 
        condition=condition, yrange=yrange)

In [ ]:
### Function to show correlations/similarities as boxplots

def show_prof_boxplot_single(
    ecto_profs_mean, myel_profs_mean,
    metric, group_labels, 
    shifts=[-1, 0, 1], condition="CT", yrange="tight", x_breaks=[],
):

    # Prep
    fig, ax = plt.subplots(1, 2, figsize=(5, 4), sharey=yrange!="indiv")
    colors = {
        "ectoderm" : plt.rcParams["axes.prop_cycle"].by_key()['color'][0],
        "myeloid"  : plt.rcParams["axes.prop_cycle"].by_key()['color'][1]
    }
    
    # Adapt for single plot
    cond = condition

    # For each cell type...
    for axis, ctype in zip(ax, colors.keys()):
        
        # Select plot data
        plot_data = []
        plot_source = {
            "ectoderm" : ecto_profs_mean, "myeloid" : myel_profs_mean}[ctype]
        for shift in shifts:
            plot_data.append(
                [plot_source[cond][p][metric].loc[shift] 
                 for p in plot_source[cond].keys()
                 if not np.isnan(plot_source[cond][p][metric].loc[shift])])

        # Create boxplot
        bp = axis.boxplot(plot_data, widths=0.6, sym='', patch_artist=True)

        # Style boxplot
        for patch in bp['boxes']:
            patch.set(color=colors[ctype], alpha=0.5)
        for whisker in bp['whiskers']:
            whisker.set(color='black', linewidth=1.2, linestyle='-', alpha=0.5)
        for cap in bp['caps']:
            cap.set(linewidth=1.2, alpha=0.6)
        for median in bp['medians']:
            median.set(color='black', linewidth=1.2, alpha=0.5)

        # Add jittered data
        for i,p in enumerate(plot_data):
            y = p
            jitter_sigma = 0.10
            x = np.random.normal(i+1, jitter_sigma, size=len(y))
            x[x>(i+1+2*jitter_sigma)] = i+1+2*jitter_sigma
            x[x<(i+1-2*jitter_sigma)] = i+1-2*jitter_sigma
            axis.plot(x, y, '.', color=colors[ctype], markeredgecolor='k', alpha=0.7, ms=8)

        # Subplot cosmetics
        axis.set_xticklabels(np.array(shifts)*time_res, fontsize=8)
        axis.set_xlabel("shift [min]")
        axis.set_title(cond_dict[cond] + "\n" + ctype, fontsize=10.5)
        
    # Global cosmetics
    pub_ylbls = {
        "cos_sim" : "cosine similarity", "speed_corr" : "correlation of speeds",
        "traj_ortho_corr" : "correlation of trajectory deviations"}
    ax[0].set_ylabel(pub_ylbls[metric] if metric in pub_ylbls else metric)
    if yrange == "tight":
        ax[0].set_ylim(yranges_tight[metric])
    if yrange == "limits":
        ax[0].set_ylim(yranges_limits[metric])
        
    # Broken axes
    for break_x in x_breaks:
        for axis in ax:
            break_y, ymax = axis.get_ylim()
            axis.scatter(break_x, break_y, color='white', marker='s', s=80, clip_on=False, zorder=100)
            axis.text(break_x+0.050, break_y-0.002, r'//', fontsize=9, zorder=101,
                      horizontalalignment='center', verticalalignment='center')
            axis.set_ylim(break_y, ymax)  # (Somehow needed only for yrange="indiv")
    
    # Done
    plt.tight_layout()

In [ ]:
### Show the correlation/similarity boxplots; boxplots with shifts around zero

shifts = [-20, -2, -1, 0, 1, 2, 20]

@interact(
    metric=metrics[2:], 
    condition=list(ecto_profs_mean.keys()), 
    yrange=["tight", "limits", "indiv"])
@savebutton
def wrapper(
    metric=metrics[2], condition="CT", yrange="tight"):
    
    group_labels = ['Ectoderm', 'Myeloid']
    show_prof_boxplot_single(
        ecto_profs_mean, myel_profs_mean, metric, group_labels, 
        shifts=shifts, condition=condition, yrange=yrange, x_breaks=[1.5, 6.5])

In [ ]:
### Get p-values for comparisons of interest

def get_profile_pval(cnd1, cnd2, ctp1, ctp2, sft1, sft2, metric, ptest="MWU"):
        
    # Get data
    data_sources = {"ectoderm" : ecto_profs_mean, "myeloid" : myel_profs_mean}
    ds1 = data_sources[ctp1]
    ds2 = data_sources[ctp2]
    d1 = [ds1[cnd1][p][metric].loc[sft1] for p in ds1[cnd1].keys()
          if not np.isnan(ds1[cnd1][p][metric].loc[sft1])]
    d2 = [ds2[cnd2][p][metric].loc[sft2] for p in ds2[cnd2].keys()
          if not np.isnan(ds2[cnd2][p][metric].loc[sft2])]
    
    # Compute p-value
    if ptest == "MWU":  # Unpaired
        stat, pval = stats.mannwhitneyu(d1, d2)
    elif ptest == "Wil":  # Paired
        stat, pval = stats.wilcoxon(d1, d2)
    
    return pval

# P-values between shift -30 and 0
shift0, shift1 = -20, {"ectoderm" : 0, "myeloid" : -1}
if flip_shifts:
    shift0, shift1 = -shift0, {k:-v for k,v in shift1.items()}
title = f"P-values between shift {shift0*time_res}min and the peak:"
print(title + "\n" + "-" * len(title))
for ctype in ["ectoderm", "myeloid"]:
    subtitle = f"{ctype} (peak shift = {shift1[ctype]*time_res}min)"
    print(f"\n  {subtitle}:" + "\n  " + "~" * (len(subtitle)+1))
    
    for cond in cond_dict:
        print(f"\n    {cond_dict[cond]}:")

        for metric in metrics_correlation[2:] + ["cos_sim"]:
            pval = get_profile_pval(
                cond, cond, ctype, ctype, shift0, shift1[ctype], metric, 
                #ptest="MWU",
                ptest="Wil",  # Swap to paired, since these comparisons *are* paired
            )
            print(f"      {metric:20} p={pval:.5f}")

# P-values between shift -1 and 0
shift0, shift1 = -1, 0
if flip_shifts:
    shift0, shift1 = -shift0, -shift1
title = f"P-values between shift {shift0*time_res}min and shift {shift1*time_res}min:"
print("\n")
print(title + "\n" + "-" * len(title))
for ctype in ["ectoderm", "myeloid"]:
    print(f"\n  {ctype}:" + "\n  " + "~" * (len(ctype)+1))
    
    for cond in cond_dict:
        print(f"\n    {cond_dict[cond]}:")

        for metric in metrics_correlation[2:] + ["cos_sim"]:
            pval = get_profile_pval(
                cond, cond, ctype, ctype, shift0, shift1, metric, 
                #ptest="MWU",
                ptest="Wil",  # Swap to paired, since these comparisons *are* paired
            )
            print(f"      {metric:20} p={pval:.5f}")

### Plot ectoderm-myeloid transfer functions for cue saturation model

In [ ]:
### Subselect a clean dataset

# Note: r_mask removed because it is not relevant in vivo

myel_cue_df = {}

for c in myel_data:
    myel_cue_df[c] = {}
    
    for p in myel_data[c]:
        
        # Time range mask
        f_mask = (myel_data[c][p]["f"] > pmr[0]) & (myel_data[c][p]["f"] < pmr[1])
        
        # True-zero velocity mask...
        z_mask = myel_data[c][p]["v_mag"] == 0.0
        
        # Pick relevant columns data
        columns = [
            "f", "t", "vy", "vx", "vy_n", "vx_n", "vy_itp", "vx_itp", 
            "v_mag", "v_itp_mag", 
            #"vx_ray", "vx_itp_ray", "vx_ray_n"  # Not available in vivo
        ]
        
        # Select & copy data
        myel_cue_df[c][p] = myel_data[c][p].loc[f_mask, columns].copy()
        
        # Add cosine distance...
        myel_cue_df[c][p]["cos_dist"] = 1.0 - myel_similarities[c][p].loc[f_mask, "cos_sim-shift=0"]
        
        # Also add cosine similarity...
        myel_cue_df[c][p]["cos_sim"] = myel_similarities[c][p].loc[f_mask, "cos_sim-shift=0"]

In [ ]:
### Compute relationship between local speeds

# Note: Adapted to enforce >=5 samples per average for greater robustness on noisy in vivo data

# Prep
c = "CT"
min_nsamples = 5
all_ecto = []
all_myel = []
xlims = (0.0, 0.5)
plt.figure(figsize=(5, 3))

# For each position/cluster...
for i,p in enumerate(myel_cue_df[c]):
    
    # Bin ectoderm velocity magnitude
    myel_cue_df[c][p]["ecto_bins"] = pd.cut(
        myel_cue_df[c][p]["v_itp_mag"], 
        bins=np.linspace(*xlims, 26)  # Standardized bin size
    )

    # Calculate binned averages for both
    binavg_ecto = myel_cue_df[c][p]["v_itp_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=False).mean().values
    binavg_myel = myel_cue_df[c][p]["v_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=False).mean().values
    
    # Exclude those with insufficient sample numbers
    binavg_count = (
        myel_cue_df[c][p]["v_itp_mag"].groupby(
            myel_cue_df[c][p]["ecto_bins"], observed=False).count().values &
        myel_cue_df[c][p]["v_mag"].groupby(
            myel_cue_df[c][p]["ecto_bins"], observed=False).count().values)
    nmask = binavg_count >= min_nsamples
    binavg_ecto[~nmask] = np.nan
    binavg_myel[~nmask] = np.nan
    
    # Plot the relationship
    plt.plot(
        binavg_ecto, binavg_myel,
        c="tab:orange", alpha=0.35, lw=1.0,
        label="Samples" if i==0 else "_none_"
    )
    
    # Keep (relevant) data for fit
    xlims_mask = (binavg_ecto > xlims[0]) & (binavg_ecto < xlims[1]) | np.isnan(binavg_ecto)
    all_ecto.append(binavg_ecto[xlims_mask])
    all_myel.append(binavg_myel[xlims_mask])
    
# Mask for sufficient amount of data available
nmask = (~(np.isnan(all_ecto) | np.isnan(all_myel))).sum(axis=0) >= min_nsamples
    
# Add overall mean
plt.plot(
    np.nanmean(np.array(all_ecto), axis=0)[nmask], 
    np.nanmean(np.array(all_myel), axis=0)[nmask], 
    c="tab:orange", lw=2.0, alpha=1.0,
    label="Mean"
)

# Legend
plt.legend(fontsize=9, frameon=False, loc=2)

# Axis limits
plt.xlim(*xlims)
plt.ylim(0.0, 3.5)

# Labels
plt.xlabel(
    r"Ectoderm flow speed $|\mathbf{v}_e|$ $[\mu m/min]$", 
    fontsize=10
)
plt.ylabel(
    r"Myeloid cell speed $|\mathbf{v}_m|$ $[\mu m/min]$", 
    fontsize=10
)

## Save for publication
#plt.title(" ")
#plt.savefig(
#    r"..\Figures\_Revision 1\in-vivo_CT_myel-speed-vs-ecto-speed.pdf", 
#    transparent=True, bbox_inches="tight"
#)

# Show figure
plt.show()

In [ ]:
### Cosine similiarity over local speed

# Note: Adapted to enforce >=5 samples per average for greater robustness on noisy in vivo data

# Prep
c = "CT"
min_nsamples = 5
all_ecto = []
all_myel = []
xlims = (0.0, 0.5)
plt.figure(figsize=(5, 3))

# For each position/cluster...
for i,p in enumerate(myel_cue_df[c]):
    
    # Bin ectoderm velocity magnitude
    myel_cue_df[c][p]["ecto_bins"] = pd.cut(
        myel_cue_df[c][p]["v_itp_mag"], 
        bins=np.linspace(*xlims, 26)  # Standardized bin size
    )

    # Calculate binned averages for both
    binavg_ecto = myel_cue_df[c][p]["v_itp_mag"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=False).mean().values
    binavg_myel = myel_cue_df[c][p]["cos_sim"].groupby(
        myel_cue_df[c][p]["ecto_bins"], observed=False).mean().values
    
    # Exclude those with insufficient sample numbers
    binavg_count = (
        myel_cue_df[c][p]["v_itp_mag"].groupby(
            myel_cue_df[c][p]["ecto_bins"], observed=False).count().values &
        myel_cue_df[c][p]["cos_sim"].groupby(
            myel_cue_df[c][p]["ecto_bins"], observed=False).count().values)
    nmask = binavg_count >= min_nsamples
    binavg_ecto[~nmask] = np.nan
    binavg_myel[~nmask] = np.nan
    
    # Plot the relationship
    plt.plot(
        binavg_ecto, binavg_myel,
        c="tab:orange", alpha=0.35, lw=1.0,
        label="Samples" if i==0 else "_none_"
    )
    
    # Keep (relevant) data for fit
    xlims_mask = (binavg_ecto > xlims[0]) & (binavg_ecto < xlims[1]) | np.isnan(binavg_ecto)
    all_ecto.append(binavg_ecto[xlims_mask])
    all_myel.append(binavg_myel[xlims_mask])

# Mask for sufficient amount of data available
nmask = (~(np.isnan(all_ecto) | np.isnan(all_myel))).sum(axis=0) >= min_nsamples
    
# Add overall mean   
plt.plot(
    np.nanmean(np.array(all_ecto), axis=0)[nmask], 
    np.nanmean(np.array(all_myel), axis=0)[nmask], 
    c="tab:orange", lw=2.0, alpha=1.0,
    label="Mean"
)

# Legend
plt.legend(fontsize=9, frameon=False, loc=2)

# Axis limits
plt.xlim(*xlims)
plt.ylim(-0.5, 1.0)

# Labels
plt.xlabel(
    r"Ectoderm flow speed $|\mathbf{v}_e|$ $[\mu m/min]$", 
    fontsize=10
)
plt.ylabel(
    "Flow-alignment of myeloid cells\n(cosine similarity)", 
    fontsize=10
)

## Save for publication
#plt.title(" ")
#plt.savefig(
#    r"..\Figures\_Revision 1\in-vivo_CT_myel-cossim-vs-ecto-speed.pdf",
#    transparent=True, bbox_inches="tight"
#)

# Show figure
plt.show()